# Failure Case Analysis

After running `python -m src.evaluate ...`, this notebook digs into the per-patient Dice distribution and surfaces the worst predictions for visual inspection.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.augmentations import eval_transform
from src.dataset import LGGSegmentationDataset, discover_slices, patient_level_split
from src.models import build_model
from src.postprocess import postprocess_mask

In [ ]:
RESULTS = Path('../results')
STEM = 'unet_resnet34_best'
SPLIT = 'test'

df_slice = pd.read_csv(RESULTS / f'{STEM}_{SPLIT}_slice.csv')
df_patient = pd.read_csv(RESULTS / f'{STEM}_{SPLIT}_patient.csv')
df_patient = df_patient.sort_values('dice')
df_patient.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_patient['dice'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(df_patient['dice'].mean(), color='red', linestyle='--', label=f"mean={df_patient['dice'].mean():.3f}")
ax.axvline(df_patient['dice'].median(), color='green', linestyle='--', label=f"median={df_patient['dice'].median():.3f}")
ax.set_xlabel('Per-patient Dice')
ax.set_ylabel('# patients')
ax.legend()
ax.set_title(f'{STEM} on {SPLIT}')

## Worst slices by Dice (with ground-truth tumor)

In [ ]:
worst = df_slice[df_slice['has_gt']].sort_values('dice').head(8)
worst[['patient_id', 'slice_idx', 'dice', 'iou', 'sensitivity']]

In [ ]:
DATA_ROOT = Path('../data/lgg-mri-segmentation/kaggle_3m')
CKPT = Path('../models') / f'{STEM}.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load(CKPT, map_location=device)
model = build_model(ckpt['model_name']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

records = discover_slices(DATA_ROOT)
by_key = {(r.patient_id, r.slice_idx): r for r in records}
tfm = eval_transform()

In [ ]:
fig, axes = plt.subplots(len(worst), 3, figsize=(11, 3.4 * len(worst)))
if len(worst) == 1:
    axes = axes.reshape(1, -1)

for row, (_, r) in enumerate(worst.iterrows()):
    rec = by_key[(r['patient_id'], int(r['slice_idx']))]
    img = cv2.cvtColor(cv2.imread(rec.image_path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256, 256))
    msk = cv2.imread(rec.mask_path, cv2.IMREAD_GRAYSCALE)
    msk = (cv2.resize(msk, (256, 256), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)

    out = tfm(image=img, mask=msk)
    x = out['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    pred = postprocess_mask((prob > 0.5).astype(np.uint8))

    axes[row, 0].imshow(img); axes[row, 0].set_title(f'{rec.patient_id[:20]}  slice {rec.slice_idx}'); axes[row, 0].axis('off')
    axes[row, 1].imshow(img); axes[row, 1].imshow(msk, alpha=0.4, cmap='Reds'); axes[row, 1].set_title('GT'); axes[row, 1].axis('off')
    axes[row, 2].imshow(img); axes[row, 2].imshow(pred, alpha=0.4, cmap='Blues'); axes[row, 2].set_title(f"pred  Dice={r['dice']:.3f}"); axes[row, 2].axis('off')
plt.tight_layout()

## Failure-mode summary
Common patterns to look for: tiny tumors with under-segmentation, off-axial slices where the encoder's ImageNet prior misfires, hyperintense non-tumor regions (false positives), and patients with motion artifacts.

In [ ]:
df_slice_gt = df_slice[df_slice['has_gt']].copy()
df_slice_gt['size_bin'] = pd.cut(df_slice_gt['dice'].rank(pct=True), bins=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df_slice_gt.groupby('size_bin', observed=True)[['dice', 'iou', 'sensitivity']].mean().round(3))